# GenAI-Kickstart
## Lesson: Augmentation

This lesson is intended to show you how to enrich prompts with internal knowledge

During this lesson you will learn ...

- how to augment prompts with external knowledge
- about the disadvantages of uploading whole, large files

### Set up the environment
Import the necessary libraries, set constants, and define helper functions.

In [1]:
import sys
sys.path.append('../../../')

In [2]:
from dotenv import load_dotenv, find_dotenv
from litellm import completion
from lib.completion import generate_completion
from lib.constants import PROCESSED_DATA_PATH
from lib.validation import validate_environment_for_model
import os

load_dotenv(find_dotenv())

True

In [3]:
# Data path imported from lib.constants
BOOK_FILE = PROCESSED_DATA_PATH / "study_in_scarlett.txt"

In [4]:
# Load the default provider and model. If you want, you can change this.
PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL")
MODEL_ID = f"{PROVIDER}/{GENERATION_MODEL}"

validate_environment_for_model(model=MODEL_ID)
print(f"Using model: {MODEL_ID}")

TEMPERATURE = 0.9
MAX_OUTPUT_TOKENS = 200
SYSTEM_PROMPT = "You are a friendly assistant"

Using model: gemini/gemini-2.5-flash


In [5]:
# Check if all necessary environment variables are set
validate_environment_for_model(model=MODEL_ID)

In [6]:
# Let's test the models knowledge on a book
user_input = "In Sherlock Holmes book 'A Study in Scarlett': Which number was written on the ceiling?"
generate_completion(user_prompt=user_input, model_name=MODEL_ID)

"In Arthur Conan Doyle's *A Study in Scarlet*, Sherlock Holmes writes the number **175** on the ceiling.\n\nHe does this as a demonstration to Watson, showing how much information he could deduce about the person who wrote the number, even with such a seemingly small detail."

### Exercise 01: Provide internal knowledge by including file content
In order to provide internal knowledge, we can include the full text of a document in the prompt. Your task is to read the BOOK_FILE and pass its content alongside the prompt. Do you find a way of getting insights into token usage?  
**Hints**:
* Read the file and include its content in the user message
* Use the `usage` attribute of the LiteLLM response to inspect token counts

In [10]:
# Question about a detailed part of the book
user_prompt = "Which number was written on the ceiling?"

# Read the full book content
book_content = BOOK_FILE.read_text()

# TODO: Build the prompt with the full file content
full_prompt = f"Based on the following book:\n\n{book_content}\n\nQuestion: {user_prompt}"

# Call the model with the full file content included
response = completion(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": full_prompt},
    ],
    temperature=TEMPERATURE,
    max_completion_tokens=MAX_OUTPUT_TOKENS,
)

# TODO: Print insights of the response's token usage
print(f"Input tokens count: {response.usage.prompt_tokens}")
print(f"Total tokens count: {response.usage.total_tokens}")
print(f"\nAnswer: {response.choices[0].message.content}")

Input tokens count: 54792
Total tokens count: 54952

Answer: In the center of the ceiling was scrawled the number **28**.


### Exercise 02: Enhance prompt with internal knowledge
Providing small pieces of context within the prompt should be the best way to answer this user query. Your task is to simply enhance the prompt below by providing the internal knowledge about the book.

In [11]:
# A paragraph from the book, which should be enough to answer the question
internal_book_knowledge_chunk = """
Still more shaken was he next morning. They had sat down to their
breakfast when Lucy with a cry of surprise pointed upwards. In the
centre of the ceiling was scrawled, with a burned stick apparently, the
number 28. To his daughter it was unintelligible, and he did not
enlighten her. That night he sat up with his gun and kept watch and
ward. He saw and he heard nothing, and yet in the morning a great 27
had been painted upon the outside of his door.
"""

user_prompt = "Which number was written on the ceiling?"

# TODO: Create an augmented prompt where the user prompt is augmented with the internal book knowledge chunk
augmented_user_prompt = f"Answer the question below based on the context.\n\nContext: {internal_book_knowledge_chunk}\n\nQuestion: {user_prompt}"

response = completion(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": augmented_user_prompt},
    ],
    temperature=TEMPERATURE,
    max_completion_tokens=MAX_OUTPUT_TOKENS,
)

# TODO: Print insights of the response's token usage
print(f"Input tokens count: {response.usage.prompt_tokens}")
print(f"Total tokens count: {response.usage.total_tokens}")
print(f"\nAnswer: {response.choices[0].message.content}")

Input tokens count: 144
Total tokens count: 265

Answer: The number written on the ceiling was **28**.


### Exercise 03: Calculate costs for both calls
Cost is one factor which should prevent us from uploading large files. But how much is it exactly? And how much more is it than just uploading small amounts of internal knowledge? Use only input tokens for calculation. Let's say we use the paid tier.  
**Hints**:  
* See [pricing](https://ai.google.dev/gemini-api/docs/pricing)

In [12]:
# TODO: Calculate the costs of both calls
def calculate_costs(token_count: int, price_per_million_tokens: float = 1.25) -> float:
    """ Calculate the costs of a given number of tokens based on the price per million tokens """
    return token_count * (price_per_million_tokens / 1_000_000)

# Token counts from the two calls above
input_token_count_file_upload = 62191
input_token_count_chunk_upload = 141

# For gemini-2.0-flash the price is 0.1 € per million tokens
euro_per_million_tokens = 0.1

costs_file_upload = calculate_costs(token_count=input_token_count_file_upload, price_per_million_tokens=euro_per_million_tokens)
costs_chunk_upload = calculate_costs(token_count=input_token_count_chunk_upload, price_per_million_tokens=euro_per_million_tokens)

print(f"Costs for {input_token_count_file_upload} tokens using file upload: {costs_file_upload:0.5f} €")
print(f"Costs for {input_token_count_chunk_upload} tokens using chunk upload: {costs_chunk_upload:0.5f} €")
print(f"Uploading the file is {round(costs_file_upload/costs_chunk_upload)} times more expensive than uploading the chunk.")

Costs for 62191 tokens using file upload: 0.00622 €
Costs for 141 tokens using chunk upload: 0.00001 €
Uploading the file is 441 times more expensive than uploading the chunk.
